4KSO, 1F16, 8E6Y did not have their metamorphic pair structures in the downloaded dataset.
1F16 and 8E6Y were deleted and the entry for 4KSO was replaced with 4ksoA & 5n8yG 

Reformat the Table S1 data from morpheus

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import biotite.database.rcsb as rcsb
import biotite.structure as struc
import biotite.structure.io as strucio
from tqdm.auto import tqdm

def _get_dssp_3state(array, use_dssp=True):
  """
  Calculates secondary structure using the official DSSP executable
  and reduces the 8-state classification down to a 3-state output (a, b, c).
  """
  if use_dssp:

    from biotite.application.dssp import DsspApp
    import sys
    import shutil
    
    # 1. Search for the executable name
    dssp_bin = shutil.which("mkdssp") or shutil.which("dssp")
    
    # 2. Gentle Conda hunt if standard PATH fails (e.g. inside Jupyter)
    if not dssp_bin:
      conda_bin_dir = os.path.dirname(sys.executable)
      if os.path.exists(os.path.join(conda_bin_dir, "mkdssp")):
        dssp_bin = os.path.join(conda_bin_dir, "mkdssp")
      elif os.path.exists(os.path.join(conda_bin_dir, "dssp")):
        dssp_bin = os.path.join(conda_bin_dir, "dssp")
        
    if not dssp_bin:
      raise FileNotFoundError("Neither 'mkdssp' nor 'dssp' found in PATH.")

    # Initialize and run DSSP application using the resolved binary
    app = DsspApp(array, bin_path=dssp_bin)
    app.start()
    app.join()
    
    # Check for execution errors (Allow both FINISHED and JOINED states)
    state = app.get_app_state()
    if "FINISHED" not in str(state) and "JOINED" not in str(state):
       raise RuntimeError(f"DSSP execution failed with state: {state}")
       
    sse_8state = app.get_sse()
    
    # Map 8-state to 3-state
    sse_3state = []
    for state in sse_8state:
      if state in ['H', 'G', 'I']:
        sse_3state.append('a')
      elif state in ['E', 'B']:
        sse_3state.append('b')
      else:
        sse_3state.append('c')
        
    return np.array(sse_3state)
  else:
    return struc.annotate_sse(array)

def save_secondary_structures(input_csv, cif_dir="fixed_cif_files", output_csv="residue_secondary_structures.csv"):
  # Read the training data
  df = pd.read_csv(input_csv)
  raw_ids = df['PDB_ID'].unique()
  results = []
  
  print(f"Processing {len(raw_ids)} unique structures...")
  
  for raw_id in raw_ids:
      
    # Local File Search: prioritize the 'cif_files' directory
    local_path = os.path.join(cif_dir, f"{raw_id}.cif")
    if os.path.exists(local_path):
      source = local_path
    else:
      # Fallback: Fetch BinaryCIF from RCSB
      tqdm.write('#'*50+f"\n{raw_id} is missing in {cif_dir}\n"+'#'*50)
      continue
    
    # Load structure with Entity ID metadata (extra_fields is required for _ IDs)
    array = strucio.load_structure(source)
        
    # Filter for protein atoms only
    array = array[struc.filter_amino_acids(array)]

    # Filter out insertion codes (keep only the standard empty string "")
    if "ins_code" in array.get_annotation_categories():
      array = array[array.ins_code == ""]
    
    if len(array) == 0:
      continue
      
    # Calculate Secondary Structure (a=alpha, b=beta, c=coil)
    sse = _get_dssp_3state(array, use_dssp=True)
    
    # Extract residue-level information
    starts = struc.get_residue_starts(array)
    res_ids = array.res_id[starts]
    res_names = array.res_name[starts]
    seen = set()
    for r_id, r_name, ss in zip(res_ids, res_names, sse):
      if r_id in seen:
        print(f"{raw_id} has mutiple residues at {r_id}.")
        continue
      seen.add(r_id)
      results.append({
        "Raw_Input_ID": raw_id,
        "Residue_ID": r_id,
        "Residue_Name": r_name,
        "SS_Label": ss
      })
        
    # print(f"Failed to process {raw_id}: {e}")
      
  # Export final results
  out_df = pd.DataFrame(results)
  out_df.to_csv(output_csv, index=False)
  print(f"\nFinished! Results saved to '{output_csv}'")

save_secondary_structures("Morpheus_Supplement/output_results/training_data.csv")

In [ ]:
import pandas as pd

# Standard mapping for amino acid codes
AA_MAP = {
  'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C', 'GLN': 'Q',
  'GLU': 'E', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I', 'LEU': 'L', 'LYS': 'K',
  'MET': 'M', 'PHE': 'F', 'PRO': 'P', 'SER': 'S', 'THR': 'T', 'TRP': 'W',
  'TYR': 'Y', 'VAL': 'V'
}

def transform_to_sequence_format(input_file="residue_secondary_structures.csv", output_file="pdb_secondary_structures.csv"):
  # Load the residue-level data
  df = pd.read_csv(input_file)
  
  # Convert 3-letter codes to 1-letter symbols
  # Ensure the name is a string before calling .upper() to avoid float errors
  df['AA_1'] = df['Residue_Name'].apply(lambda x: AA_MAP.get(str(x).upper(), 'X'))
  
  # Ensure the data is sorted by Residue_ID to maintain sequence order
  df = df.sort_values(by=['Raw_Input_ID', 'Residue_ID'])
  
  # Helper function to join values while ignoring NaNs and forcing string type
  def join_labels(series):
    # Filter out NaNs, convert to string, and join
    return ''.join(series.dropna().astype(str))

  # Aggregate the residues into strings
  transformed = df.groupby('Raw_Input_ID').agg({
    'AA_1': join_labels,
    'SS_Label': join_labels,
  }).reset_index()
  
  # Rename for clarity
  transformed = transformed.rename(columns={
    'AA_1': 'Sequence',
    'SS_Label': 'Secondary_Structure'
  })
  
  # Reorder columns
  cols = ['Raw_Input_ID', 'Sequence', 'Secondary_Structure']
  transformed = transformed[cols]
  
  transformed.to_csv(output_file, index=False)
  print(f"Transformation complete. Data saved to '{output_file}'")


transform_to_sequence_format()

In [ ]:
import os
import difflib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.spatial.transform import Rotation as R
from scipy.spatial.distance import cdist
import biotite.structure as struc
import biotite.structure.io as strucio

# Standard mapping for 3-letter to 1-letter amino acid codes
AA_MAP = {
  'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C', 'GLN': 'Q',
  'GLU': 'E', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I', 'LEU': 'L', 'LYS': 'K',
  'MET': 'M', 'PHE': 'F', 'PRO': 'P', 'SER': 'S', 'THR': 'T', 'TRP': 'W',
  'TYR': 'Y', 'VAL': 'V'
}

def _load_and_extract_chain(raw_id, cif_dir, pdb_dir):
  """Helper to load a structure and resolve the entity/chain ID."""
  if "_" in raw_id:
    pdb_id, entity_id = raw_id.split("_", 1)
    chain_id = None
  elif "." in raw_id:
    pdb_id, chain_id = raw_id.split(".", 1)
    entity_id = None
  else:
    pdb_id = raw_id
    chain_id = None
    entity_id = None

  source = os.path.join(cif_dir, f"{pdb_id}.cif")
  if not os.path.exists(source):
    source = os.path.join(pdb_dir, f"{pdb_id}.pdb")
    if not os.path.exists(source):
      raise FileNotFoundError(f"Missing file for {pdb_id}")

  extra = ["label_entity_id"] if entity_id else []
  array = strucio.load_structure(source, model=1, extra_fields=extra)
  
  if entity_id:
    entity_mask = (array.label_entity_id == str(entity_id))
    chain_id = np.unique(array.chain_id[entity_mask])[0]
    
  if chain_id:
    array = array[array.chain_id == chain_id]

  array = array[struc.filter_amino_acids(array)]

  # Filter out insertion codes (keep only the standard empty string "")
  if "ins_code" in array.get_annotation_categories():
    array = array[array.ins_code == ""]

  return array

# def _get_sequence_and_ids(array):
#   """Extracts the 1-letter sequence and the corresponding residue IDs."""
#   starts = struc.get_residue_starts(array)
#   res_names = array.res_name[starts]
#   res_ids = array.res_id[starts]
#   seq = "".join([AA_MAP.get(name, 'X') for name in res_names])
#   return seq, res_ids

def _get_sequence_and_ids(array):
  """Extracts the 1-letter sequence and the corresponding residue IDs."""
    
  starts = struc.get_residue_starts(array)
  
  # For microheterogeneity, we can just grab the first unique occurrence of each res_id
  res_names = array.res_name[starts]
  raw_ids = array.res_id[starts]
  
  # Deduplicate while preserving sequence order
  unique_ids = []
  unique_names = []
  seen = set()
  
  for name, r_id in zip(res_names, raw_ids):
    if r_id not in seen:
      seen.add(r_id)
      unique_ids.append(r_id)
      unique_names.append(name)
      
  seq = "".join([AA_MAP.get(name, 'X') for name in unique_names])
  
  return seq, np.array(unique_ids)

def _calculate_steric_bounds_for_array(array, target_res_ids, local_window=2, clash_cutoff=2.0, global_check=False):
  """Calculates the CW and ACW allowance arrays for a specific list of residues."""
  phi_bounds = []
  psi_bounds = []

  def find_max_rotation(axis_vec, origin, moving_coords, static_coords, ignore_mask):
    max_cw, max_acw = 0, 0
    vec = axis_vec / np.linalg.norm(axis_vec)
    for deg in range(1, 181):
      rot = R.from_rotvec(np.deg2rad(deg) * vec)
      rotated = rot.apply(moving_coords - origin) + origin
      if np.any((cdist(rotated, static_coords) < clash_cutoff) & (~ignore_mask)):
        break
      max_cw = deg
    for deg in range(-1, -181, -1):
      rot = R.from_rotvec(np.deg2rad(deg) * vec)
      rotated = rot.apply(moving_coords - origin) + origin
      if np.any((cdist(rotated, static_coords) < clash_cutoff) & (~ignore_mask)):
        break
      max_acw = deg
    return max_cw, max_acw

  for res_id in target_res_ids:
    res_mask = (array.res_id == res_id)
    scope_mask = np.ones(len(array), dtype=bool) if global_check else (array.res_id >= res_id - local_window) & (array.res_id <= res_id + local_window)
    
    bounds = {'phi': (0, 0), 'psi': (0, 0)}
    
    for angle_type in ['phi', 'psi']:
      moving_mask = np.zeros(len(array), dtype=bool)
      static_mask = np.zeros(len(array), dtype=bool)

      if angle_type == 'phi':
        axis_atom1, axis_atom2 = 'N', 'CA'
        static_mask = scope_mask & (array.res_id < res_id)
        n_idx = np.where(res_mask & (array.atom_name == 'N'))[0]
        if len(n_idx) > 0: static_mask[n_idx[0]] = True
        moving_mask = scope_mask & ~static_mask

      elif angle_type == 'psi':
        axis_atom1, axis_atom2 = 'CA', 'C'
        moving_mask = scope_mask & (array.res_id > res_id)
        c_idx = np.where(res_mask & (array.atom_name == 'C'))[0]
        o_idx = np.where(res_mask & (array.atom_name == 'O'))[0]
        if len(c_idx) > 0: moving_mask[c_idx[0]] = True
        if len(o_idx) > 0: moving_mask[o_idx[0]] = True
        static_mask = scope_mask & ~moving_mask

      a1 = array[res_mask & (array.atom_name == axis_atom1)]
      a2 = array[res_mask & (array.atom_name == axis_atom2)]
      
      if len(a1) > 0 and len(a2) > 0:
        origin = a1.coord[0]
        axis_vec = a2.coord[0] - origin
        moving_coords = array.coord[moving_mask]
        static_coords = array.coord[static_mask]
        
        if len(moving_coords) > 0 and len(static_coords) > 0:
          initial_dists = cdist(moving_coords, static_coords)
          ignore_mask = initial_dists < clash_cutoff
          cw, acw = find_max_rotation(axis_vec, origin, moving_coords, static_coords, ignore_mask)
          bounds[angle_type] = (cw, acw)
          
    phi_bounds.append(bounds['phi'])
    psi_bounds.append(bounds['psi'])

  phi_err = np.array([[np.abs(acw), cw] for cw, acw in phi_bounds]).T
  psi_err = np.array([[np.abs(acw), cw] for cw, acw in psi_bounds]).T
  
  return phi_err, psi_err

def plot_comparative_dihedrals_with_errors(id1, id2, cif_dir="cif_files", pdb_dir="pdb_files", local_window=2, global_check=False, use_dssp=True):
  try:
    print(f"Loading structures {id1} and {id2}...")
    arr1 = _load_and_extract_chain(id1, cif_dir, pdb_dir)
    arr2 = _load_and_extract_chain(id2, cif_dir, pdb_dir)

    seq1, ids1 = _get_sequence_and_ids(arr1)
    seq2, ids2 = _get_sequence_and_ids(arr2)

    # Calculate SSE on the full array using the clean DSSP binary
    print("Annotating Secondary Structure ...")
    sse1_full = _get_dssp_3state(arr1, use_dssp)
    sse2_full = _get_dssp_3state(arr2, use_dssp)

    matcher = difflib.SequenceMatcher(None, seq1, seq2, autojunk=False)
    match = matcher.find_longest_match(0, len(seq1), 0, len(seq2))
    
    if match.size == 0:
      print("No common sequence found.")
      return

    match_ids1 = ids1[match.a : match.a + match.size]
    match_ids2 = ids2[match.b : match.b + match.size]
    
    sse1_match = sse1_full[match.a : match.a + match.size]
    sse2_match = sse2_full[match.b : match.b + match.size]
    
    sub_arr1 = arr1[np.isin(arr1.res_id, match_ids1)]
    sub_arr2 = arr2[np.isin(arr2.res_id, match_ids2)]

    phi1, psi1, _ = struc.dihedral_backbone(sub_arr1)
    phi2, psi2, _ = struc.dihedral_backbone(sub_arr2)
    phi1, psi1 = np.rad2deg(phi1), np.rad2deg(psi1)
    phi2, psi2 = np.rad2deg(phi2), np.rad2deg(psi2)

    print(f"Calculating steric bounds for {match.size} residues... (Global Check: {global_check})")
    phi1_err, psi1_err = _calculate_steric_bounds_for_array(arr1, match_ids1, local_window, 2.0, global_check)
    phi2_err, psi2_err = _calculate_steric_bounds_for_array(arr2, match_ids2, local_window, 2.0, global_check)

    x_axis = np.arange(match.size)
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(0.26*match.size, 11), sharex=True, gridspec_kw={'height_ratios': [4, 4, 1.5]})
    
    title_str = (
      f"Metamorphic Freedom & SSE: {id1} vs {id2}\n"
      f"LCS Length: {match.size} AA | "
      f"{id1} (Res {match_ids1[0]}-{match_ids1[-1]}) | {id2} (Res {match_ids2[0]}-{match_ids2[-1]})"
    )
    
    offset = 0.15 
    
    # 1. Phi
    ax1.errorbar(x_axis - offset, phi1, yerr=phi1_err, fmt='o', color='tab:blue', ecolor='tab:blue', 
                 capsize=3, elinewidth=1.5, alpha=0.8, label=f"{id1}")
    ax1.errorbar(x_axis + offset, phi2, yerr=phi2_err, fmt='s', color='tab:cyan', ecolor='tab:cyan', 
                 capsize=3, elinewidth=1.5, alpha=0.8, label=f"{id2}")
    ax1.set_title(title_str, fontweight='bold')
    ax1.set_ylabel(r"$\phi$ (Phi) Angle (Deg)")
    # ax1.set_ylim(-180, 180)
    ax1.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax1.grid(alpha=0.3)
    ax1.legend()

    # 2. Psi
    ax2.errorbar(x_axis - offset, psi1, yerr=psi1_err, fmt='o', color='tab:red', ecolor='tab:red', 
                 capsize=3, elinewidth=1.5, alpha=0.8, label=f"{id1}")
    ax2.errorbar(x_axis + offset, psi2, yerr=psi2_err, fmt='s', color='tab:orange', ecolor='tab:orange', 
                 capsize=3, elinewidth=1.5, alpha=0.8, label=f"{id2}")
    ax2.set_ylabel(r"$\psi$ (Psi) Angle (Deg)")
    # ax2.set_ylim(-180, 180)
    ax2.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax2.grid(alpha=0.3)
    ax2.legend()

    # 3. Secondary Structure
    sse_colors = {'a': '#ff4c4c', 'b': '#4c4cff', 'c': '#cccccc'}
    
    for i in range(match.size):
      ax3.barh(1, 1, left=i-0.5, color=sse_colors.get(sse1_match[i], '#cccccc'), edgecolor='none', height=0.6)
      ax3.barh(0, 1, left=i-0.5, color=sse_colors.get(sse2_match[i], '#cccccc'), edgecolor='none', height=0.6)
      
    ax3.set_yticks([0, 1])
    ax3.set_yticklabels([id2, id1])
    ax3.set_ylim(-0.5, 1.5)
    ax3.set_ylabel("SSE")
    ax3.set_xlabel("Shared Alignment Index")
    
    legend_elements = [
      Patch(facecolor='#ff4c4c', label=r'$\alpha$-Helix'),
      Patch(facecolor='#4c4cff', label=r'$\beta$-Sheet'),
      Patch(facecolor='#cccccc', label='Coil')
    ]
    ax3.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1.0, 1.0))
    ax3.grid(alpha=0.3, axis='x')
    ax3.set_xticks(x_axis)
    ax3.tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()

  except Exception as e:
    print(f"Error analyzing {id1} and {id2}: {e}")

# Example Usage:
# plot_comparative_dihedrals_with_errors("5JYT.A", "2QKE.E", local_window=2, global_check=False)

In [ ]:
def id_convert_func(raw_id):
  raw_id = raw_id.upper()
  if raw_id[-1].isdigit():
    pdb_id = raw_id[:-1]+'_'+raw_id[-1]
  else:
    pdb_id = raw_id[:-1]+'.'+raw_id[-1]
  return pdb_id
pdb_pair_df = pd.read_csv('metamorphic_pairs.csv')

In [ ]:
for idx in range(11):
  pdbid1 = id_convert_func(pdb_pair_df.iloc[idx]['PDB1'])
  pdbid2 = id_convert_func(pdb_pair_df.iloc[idx]['PDB2'])
  plot_comparative_dihedrals_with_errors(pdbid1, pdbid2, global_check=True)

Some observations:
 - In 1K0N.A and 1RK4.B, the LCS is much shorter than the actual sequence because an E at a given position in one structure is a Q in another.
 - Thus, the tertiary contacts of positions within the LCS are often outside the LCS. 

In [ ]:
def plot_dihedrals_with_errors(pdb_id, cif_dir="cif_files", pdb_dir="pdb_files", local_window=2, global_check=False, use_dssp=True):
  print(f"Loading structure {pdb_id}...")
  arr = _load_and_extract_chain(pdb_id, cif_dir, pdb_dir)

  seq, ids = _get_sequence_and_ids(arr)

  # Calculate SSE on the full array using the clean DSSP binary
  sse_full = _get_dssp_3state(arr, use_dssp)

  phi, psi, _ = struc.dihedral_backbone(arr)
  phi, psi = np.rad2deg(phi), np.rad2deg(psi)

  print(f"Calculating steric bounds for {len(seq)} residues... (Global Check: {global_check})")
  phi_err, psi_err = _calculate_steric_bounds_for_array(arr, ids, local_window, 2.0, global_check)

  x_axis = np.arange(len(seq))
  fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(0.24*len(seq), 11), sharex=True, gridspec_kw={'height_ratios': [4, 4, 0.8]})
  
  title_str = (
    f"Metamorphic Freedom & SSE: {pdb_id}\n"
    f"Sequence Length: {len(seq)} AA "
  )
  
  # 1. Phi
  ax1.errorbar(x_axis, phi, yerr=phi_err, fmt='o', color='tab:blue', ecolor='tab:blue', 
               capsize=3, elinewidth=1.5, alpha=0.8, label=f"{pdb_id}")
  ax1.set_title(title_str, fontweight='bold')
  ax1.set_ylabel(r"$\phi$ (Phi) Angle (Deg)")
  ax1.set_ylim(-180, 180)
  ax1.axhline(0, color='black', linewidth=0.5, linestyle='--')
  ax1.grid(alpha=0.3)
  ax1.legend()

  # 2. Psi
  ax2.errorbar(x_axis, psi, yerr=psi_err, fmt='o', color='tab:red', ecolor='tab:red', 
               capsize=3, elinewidth=1.5, alpha=0.8, label=f"{pdb_id}")
  ax2.set_ylabel(r"$\psi$ (Psi) Angle (Deg)")
  ax2.set_ylim(-180, 180)
  ax2.axhline(0, color='black', linewidth=0.5, linestyle='--')
  ax2.grid(alpha=0.3)
  ax2.legend()

  # 3. Secondary Structure
  sse_colors = {'a': '#ff4c4c', 'b': '#4c4cff', 'c': '#cccccc'}
  
  for i in range(len(seq)):
    ax3.barh(0, 1, left=i-0.5, color=sse_colors.get(sse_full[i], '#cccccc'), edgecolor='none', height=0.6)
    
  ax3.set_yticks([0])
  ax3.set_yticklabels([pdb_id])
  ax3.set_ylim(-0.5, 0.6)
  ax3.set_ylabel("SSE")
  ax3.set_xlabel("Shared Alignment Index")
  
  legend_elements = [
    Patch(facecolor='#ff4c4c', label=r'$\alpha$-Helix'),
    Patch(facecolor='#4c4cff', label=r'$\beta$-Sheet'),
    Patch(facecolor='#cccccc', label='Coil')
  ]
  ax3.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1.0, 1.0))
  ax3.grid(alpha=0.3, axis='x')
  ax3.set_xticks(x_axis)
  ax3.tick_params(axis='x', rotation=45)

  plt.tight_layout()
  plt.show()


In [ ]:
plot_dihedrals_with_errors("1OVA.A", global_check=True)

# Create datasets

In [ ]:
import os
import difflib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.spatial.transform import Rotation as R
from scipy.spatial.distance import cdist
import biotite.structure as struc
import biotite.structure.io as strucio

# Standard mapping for 3-letter to 1-letter amino acid codes
AA_MAP = {
  'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C', 'GLN': 'Q',
  'GLU': 'E', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I', 'LEU': 'L', 'LYS': 'K',
  'MET': 'M', 'PHE': 'F', 'PRO': 'P', 'SER': 'S', 'THR': 'T', 'TRP': 'W',
  'TYR': 'Y', 'VAL': 'V'
}

def _load_and_extract_chain(raw_id, cif_dir):
  """Helper to load a structure and resolve the entity/chain ID."""

  source = os.path.join(cif_dir, f"{raw_id}.cif")
  if not os.path.exists(source):
    raise FileNotFoundError(f"Missing file for {raw_id}")

  array = strucio.load_structure(source)
  
  array = array[struc.filter_amino_acids(array)]

  # Filter out insertion codes (keep only the standard empty string "")
  if "ins_code" in array.get_annotation_categories():
    array = array[array.ins_code == ""]

  return array

# def _get_sequence_and_ids(array):
#   """Extracts the 1-letter sequence and the corresponding residue IDs."""
#   starts = struc.get_residue_starts(array)
#   res_names = array.res_name[starts]
#   res_ids = array.res_id[starts]
#   seq = "".join([AA_MAP.get(name, 'X') for name in res_names])
#   return seq, res_ids

def _get_sequence_and_ids(array):
  """Extracts the 1-letter sequence and the corresponding residue IDs."""
    
  starts = struc.get_residue_starts(array)
  
  # For microheterogeneity, we can just grab the first unique occurrence of each res_id
  res_names = array.res_name[starts]
  res_ids = array.res_id[starts]
  
  # Deduplicate while preserving sequence order
  unique_ids = []
  unique_names = []
  seen = set()
  
  for name, r_id in zip(res_names, res_ids):
    if r_id not in seen:
      seen.add(r_id)
      unique_ids.append(r_id)
      unique_names.append(name)
      
  seq = "".join([AA_MAP.get(name, 'X') for name in unique_names])
  
  return seq, np.array(unique_ids)

def _calculate_steric_bounds_for_array(array, target_res_ids, local_window=2, clash_cutoff=2.0, global_check=False):
  """Calculates the CW and ACW allowance arrays for a specific list of residues."""
  phi_bounds = []
  psi_bounds = []

  def find_max_rotation(axis_vec, origin, moving_coords, static_coords, ignore_mask):
    max_cw, max_acw = 0, 0
    vec = axis_vec / np.linalg.norm(axis_vec)
    for deg in range(1, 181):
      rot = R.from_rotvec(np.deg2rad(deg) * vec)
      rotated = rot.apply(moving_coords - origin) + origin
      if np.any((cdist(rotated, static_coords) < clash_cutoff) & (~ignore_mask)):
        break
      max_cw = deg
    for deg in range(-1, -181, -1):
      rot = R.from_rotvec(np.deg2rad(deg) * vec)
      rotated = rot.apply(moving_coords - origin) + origin
      if np.any((cdist(rotated, static_coords) < clash_cutoff) & (~ignore_mask)):
        break
      max_acw = deg
    return max_cw, max_acw

  for res_id in target_res_ids:
    res_mask = (array.res_id == res_id)
    scope_mask = np.ones(len(array), dtype=bool) if global_check else (array.res_id >= res_id - local_window) & (array.res_id <= res_id + local_window)
    
    bounds = {'phi': (0, 0), 'psi': (0, 0)}
    
    for angle_type in ['phi', 'psi']:
      moving_mask = np.zeros(len(array), dtype=bool)
      static_mask = np.zeros(len(array), dtype=bool)

      if angle_type == 'phi':
        axis_atom1, axis_atom2 = 'N', 'CA'
        static_mask = scope_mask & (array.res_id < res_id)
        n_idx = np.where(res_mask & (array.atom_name == 'N'))[0]
        if len(n_idx) > 0: static_mask[n_idx[0]] = True
        moving_mask = scope_mask & ~static_mask

      elif angle_type == 'psi':
        axis_atom1, axis_atom2 = 'CA', 'C'
        moving_mask = scope_mask & (array.res_id > res_id)
        c_idx = np.where(res_mask & (array.atom_name == 'C'))[0]
        o_idx = np.where(res_mask & (array.atom_name == 'O'))[0]
        if len(c_idx) > 0: moving_mask[c_idx[0]] = True
        if len(o_idx) > 0: moving_mask[o_idx[0]] = True
        static_mask = scope_mask & ~moving_mask

      a1 = array[res_mask & (array.atom_name == axis_atom1)]
      a2 = array[res_mask & (array.atom_name == axis_atom2)]
      
      if len(a1) > 0 and len(a2) > 0:
        origin = a1.coord[0]
        axis_vec = a2.coord[0] - origin
        moving_coords = array.coord[moving_mask]
        static_coords = array.coord[static_mask]
        
        if len(moving_coords) > 0 and len(static_coords) > 0:
          initial_dists = cdist(moving_coords, static_coords)
          ignore_mask = initial_dists < clash_cutoff
          cw, acw = find_max_rotation(axis_vec, origin, moving_coords, static_coords, ignore_mask)
          bounds[angle_type] = (cw, acw)
          
    phi_bounds.append(bounds['phi'])
    psi_bounds.append(bounds['psi'])

  phi_err = np.array([[np.abs(acw), cw] for cw, acw in phi_bounds]).T
  psi_err = np.array([[np.abs(acw), cw] for cw, acw in psi_bounds]).T
  
  return phi_err, psi_err

Let's save the dihedral angles and their rotational freedoms into a file

In [ ]:
input_csv = "residue_secondary_structures.csv"
output_csv = "residue_dihedral_rotational_freedom.csv"
cif_dir="cif_files"
pdb_dir="pdb_files"
df = pd.read_csv(input_csv)

# Initialize the 6 new feature columns with NaNs
for col in ['phi_min', 'phi', 'phi_max', 'psi_min', 'psi', 'psi_max']:
  df[col] = np.nan

# Group by the structure ID to minimize file I/O operations
grouped = df.groupby('Raw_Input_ID')

for raw_id, group in grouped:
  print(f"Processing structure: {raw_id}...")
  # Load the filtered AtomArray using the unified loader
  arr = _load_and_extract_chain(raw_id, cif_dir, pdb_dir)
  
  # Compute the baseline dihedrals for the entire continuous chain in degrees
  phi_rad, psi_rad, _ = struc.dihedral_backbone(arr)
  phi_deg = np.rad2deg(phi_rad)
  psi_deg = np.rad2deg(psi_rad)
  
  # Extract residue IDs that match the dihedral array indices
  starts = struc.get_residue_starts(arr)
  res_ids_in_arr = arr.res_id[starts]
  
  # Create fast lookup dictionaries for the baseline angles
  phi_dict = dict(zip(res_ids_in_arr, phi_deg))
  psi_dict = dict(zip(res_ids_in_arr, psi_deg))

  # Isolate the exact residues requested in the CSV for this chain
  target_res_ids = group['Residue_ID'].values
  
  print(f"  Calculating steric bounds for {len(target_res_ids)} target residues...")
  # Calculate the allowances. Using local_window=2 and global_check=False as defaults.
  phi_err, psi_err = _calculate_steric_bounds_for_array(arr, target_res_ids, local_window=2, clash_cutoff=2.0, global_check=False)
  
  # The error arrays are shaped (2, N). Index 0 is the absolute ACW value, Index 1 is the CW value.
  abs_acw_phi, cw_phi = phi_err[0], phi_err[1]
  abs_acw_psi, cw_psi = psi_err[0], psi_err[1]

  # Map the computed bounds back into the correct dataframe rows
  for idx, res_id in enumerate(target_res_ids):
    # Locate the exact row index for this specific residue
    df_idx = group.index[group['Residue_ID'] == res_id].tolist()[0]
    
    # Retrieve baseline angles (Biotite returns NaN for chain termini)
    curr_phi = phi_dict.get(res_id, np.nan)
    curr_psi = psi_dict.get(res_id, np.nan)
    
    # Populate the DataFrame if the baseline angle exists
    if not np.isnan(curr_phi):
      df.at[df_idx, 'phi'] = curr_phi
      df.at[df_idx, 'phi_min'] = curr_phi - abs_acw_phi[idx]
      df.at[df_idx, 'phi_max'] = curr_phi + cw_phi[idx]
      
    if not np.isnan(curr_psi):
      df.at[df_idx, 'psi'] = curr_psi
      df.at[df_idx, 'psi_min'] = curr_psi - abs_acw_psi[idx]
      df.at[df_idx, 'psi_max'] = curr_psi + cw_psi[idx]

# Save the finalized feature matrix
df.to_csv(output_csv, index=False)
print(f"\nFeature engineering complete! Enriched dataset saved to: {output_csv}")

save fold-switch labels

In [ ]:
AA_MAP = {
  'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C', 'GLN': 'Q',
  'GLU': 'E', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I', 'LEU': 'L', 'LYS': 'K',
  'MET': 'M', 'PHE': 'F', 'PRO': 'P', 'SER': 'S', 'THR': 'T', 'TRP': 'W',
  'TYR': 'Y', 'VAL': 'V'
}

def format_pdb_id(raw_str):
  """Converts '1ceeB' format to '1CEE.B' to match Raw_Input_ID."""
  raw_str = str(raw_str).strip()
  if len(raw_str) >= 5:
    return f"{raw_str[:4].upper()}.{raw_str[4:].upper()}"
  return raw_str.upper()

struct_csv = "residue_secondary_structures.csv"
pairs_csv = "metamorphic_pairs.csv"
output_csv = "residue_switch_labels.csv"


print(f"Loading pairs from {pairs_csv}...")
pairs_df = pd.read_csv(pairs_csv)

# Create a two-way dictionary for instant partner lookups
pair_map = {}
for _, row in pairs_df.iterrows():
  p1 = format_pdb_id(row['PDB1'])
  p2 = format_pdb_id(row['PDB2'])
  pair_map[p1] = p2
  pair_map[p2] = p1

print(f"Loading structures dataset from {struct_csv}...")
df = pd.read_csv(struct_csv)

# Initialize the new column with the default SS_Label 
df['switch_SS_label'] = df['SS_Label']

print("Extracting sequences directly from CSV data...")
data_cache = {}
grouped = df.groupby('Raw_Input_ID')

for raw_id, group in grouped:
  res_names = group['Residue_Name'].values
  ss_labels = group['SS_Label'].values
  
  # Build 1-letter sequence string for difflib matching
  seq = "".join([AA_MAP.get(name, 'X') for name in res_names])
  
  # Cache the indices and data so we can map results perfectly back to the DataFrame
  data_cache[raw_id] = {
    'seq': seq,
    'ss_labels': ss_labels,
    'df_indices': group.index.values 
  }

for raw_id, data in data_cache.items():
  if raw_id in pair_map:
    partner_id = pair_map[raw_id]
    
    # We can only map if the partner is ALSO in the CSV dataset
    if partner_id in data_cache:
      print(f"Mapping metamorphic pair: {raw_id} <-> {partner_id}")
      
      # Default all residues in this specific structure to 'x'
      df.loc[data['df_indices'], 'switch_SS_label'] = 'x'
      
      seq1 = data['seq']
      partner_data = data_cache[partner_id]
      seq2 = partner_data['seq']
      ss_labels2 = partner_data['ss_labels']
      
      # Find Longest Common Subsequence (LCS)
      matcher = difflib.SequenceMatcher(None, seq1, seq2, autojunk=False)
      match = matcher.find_longest_match(0, len(seq1), 0, len(seq2))
      
      if match.size > 0:
        # Get the exact dataframe rows for sequence 1 in the matched block
        match_df_indices = data['df_indices'][match.a : match.a + match.size]
        
        # Get the corresponding partner SS labels from sequence 2 in the matched block
        ss2_match = ss_labels2[match.b : match.b + match.size]
        
        # Fast direct injection back into the dataframe
        df.loc[match_df_indices, 'switch_SS_label'] = ss2_match
        
    else:
       print(f"Partner {partner_id} for {raw_id} not found in the CSV. Skipping.")

df.to_csv(output_csv, index=False)
print(f"\nData fusion complete! Enriched dataset saved to: {output_csv}")

In [ ]:
csv_file = "residue_switch_labels.csv"
print(f"Loading dataset from {csv_file}...")
df = pd.read_csv(csv_file)

# Define valid secondary structure labels
valid_labels = ['a', 'b', 'c']

# Filter for rows where a true switch occurred
# (ignoring 'x' which means no alignment, and ignoring non-changes)
mask = (
  (df['SS_Label'] != df['switch_SS_label']) & 
  (df['switch_SS_label'] != 'x') &
  (df['SS_Label'].isin(valid_labels)) &
  (df['switch_SS_label'].isin(valid_labels))
)

switches_df = df[mask].copy()

# Create a unified transition label (e.g., 'a_to_b')
switches_df['transition'] = switches_df['SS_Label'] + "_to_" + switches_df['switch_SS_label']

# Crucial Step: Drop duplicates to ensure we only count a transition once per structure
unique_switches = switches_df.drop_duplicates(subset=['Raw_Input_ID', 'transition'])

# Count the occurrences
transition_counts = unique_switches['transition'].value_counts().to_dict()

# Display the results
print("\nNumber of unique structures exhibiting each switch type:")
print("-" * 55)

name_map = {'a': 'Alpha Helix', 'b': 'Beta Sheet', 'c': 'Coil'}

for transition, count in transition_counts.items():
  src, dst = transition.split('_to_')
  print(f"{name_map[src]:>11}  ->  {name_map[dst]:<11} : {count} structures")

## create train/val/test splits

In [ ]:
import pandas as pd
data_df = pd.read_csv('residue_switch_labels.csv')

In [ ]:
data_dict = {}
for raw_id in data_df['Raw_Input_ID'].unique():
  id_df = data_df[data_df['Raw_Input_ID'] == raw_id]
  id_dict = {f'{ss1}/{ss2}':0 for ss1 in ['a', 'b', 'c'] for ss2 in ['a', 'b', 'c']}
  for row in id_df.iterrows():
    if 'x' in row[1]['switch_SS_label']:
      continue
    ss1 = row[1]['SS_Label']
    ss2 = row[1]['switch_SS_label']
    id_dict[f'{ss1}/{ss2}'] += 1
  data_dict[raw_id] = list(id_dict.values())

In [ ]:
label_count_arr = np.array(list(data_dict.values()))

In [ ]:
total = len(label_count_arr)
weights = label_count_arr.sum(0)/label_count_arr.sum()
scores = []
for i in range(100000):
  rng = np.random.default_rng(i)
  order = rng.permutation(total)
  train_count_arr = label_count_arr[order[:round(0.8*total)]]
  test_count_arr = label_count_arr[order[round(0.8*total):]]
  train_vec = train_count_arr.sum(0)/train_count_arr.sum()
  test_vec = test_count_arr.sum(0)/test_count_arr.sum()
  score = np.linalg.norm((train_vec - test_vec)/weights)
  scores.append(score)
np.argmax(scores), np.argmin(scores), max(scores), min(scores)

In [ ]:
total = len(label_count_arr)
scores = []
for i in range(100000):
  rng = np.random.default_rng(i)
  order = rng.permutation(total)
  train_count_arr = label_count_arr[order[:round(0.8*total)]]
  test_count_arr = label_count_arr[order[round(0.8*total):]]
  train_vec = train_count_arr.sum(0)
  train_vec = train_vec/np.linalg.norm(train_vec)
  test_vec = test_count_arr.sum(0)
  test_vec = test_vec/np.linalg.norm(test_vec)
  score = train_vec.dot(test_vec)
  scores.append(score)
np.argmax(scores), np.argmin(scores), max(scores), min(scores)

In [ ]:
i=38120
rng = np.random.default_rng(i)
order = rng.permutation(total)
train_count_arr = label_count_arr[order[:round(0.8*total)]]
test_count_arr = label_count_arr[order[round(0.8*total):]]
train_vec = train_count_arr.sum(0)
train_vec = train_vec/np.linalg.norm(train_vec)
test_vec = test_count_arr.sum(0)
test_vec = test_vec/np.linalg.norm(test_vec)
score = train_vec.dot(test_vec)
score, train_vec, test_vec